In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import json
import random
from pathlib import Path

# Change path as needed
path = Path('/content/drive/MyDrive/DPO/pre-processing/')
INPUT_CSV = path / 'generated_response_variants_3.csv'

# Edit this list to process any number of countries.
# naming:
#   US : D_syn_USA_599.jsonl and US_triplets_3594_3.jsonl
COUNTRY_CODES = ['US', 'MEX']

In [3]:
def resolve_country_codes(country_code):
    if country_code == 'US':
        return 'USA', 'US'
    return country_code, country_code

def build_country_paths(country_code):
    data_country_code, adapter_tag = resolve_country_codes(country_code)
    raw_file = path / f'D_syn_{data_country_code}_599.jsonl'
    triplets_csv = path / f'{adapter_tag}_triplets_3594_3.csv'
    triplets_jsonl = path / f'{adapter_tag}_triplets_3594_3.jsonl'
    return raw_file, triplets_csv, triplets_jsonl, data_country_code, adapter_tag

def normalize_prompt(text):
    if not isinstance(text, str) or not text:
        return text
    first = text.split()[0]
    if first != 'Context:':
        if first == 'context:':
            return text[:1].upper() + text[1:]
        return 'Context: ' + text
    return text

def normalize_country_df(country_df):
    country_df = country_df.copy()
    country_df['prompt'] = country_df['prompt'].apply(normalize_prompt)
    country_df['gps_dimension'] = country_df['gps_dimension'].replace('negrecip', 'negative_reciprocity')
    country_df['gps_dimension'] = country_df['gps_dimension'].replace('posrecip', 'positive_reciprocity')
    country_df['gps_dimension'] = country_df['gps_dimension'].replace('risktaking', 'risk_taking')
    return country_df


In [4]:
df = pd.read_csv(INPUT_CSV)

country_frames = {}
country_merge_frames = {}
country_scores = {}

for country_code in COUNTRY_CODES:
    raw_file, triplets_csv, triplets_jsonl, data_country_code, adapter_tag = build_country_paths(country_code)
    country_df = pd.read_json(raw_file, lines=True)
    country_df = normalize_country_df(country_df)

    country_frames[country_code] = country_df
    country_merge_frames[country_code] = df.merge(country_df, on='prompt', how='left')

    print(f'{country_code}: loaded {len(country_df)} rows from {raw_file}')


US: loaded 599 rows from /content/drive/MyDrive/DPO/pre-processing/D_syn_USA_599.jsonl
MEX: loaded 599 rows from /content/drive/MyDrive/DPO/pre-processing/D_syn_MEX_599.jsonl


In [5]:
# Optional sanity check: compare prompts across countries when at least two are provided.
if len(COUNTRY_CODES) >= 2:
    first_country = COUNTRY_CODES[0]
    second_country = COUNTRY_CODES[1]
    print(country_frames[first_country]['prompt'].eq(country_frames[second_country]['prompt']).value_counts())


prompt
False    599
Name: count, dtype: int64


In [6]:
PAIR_LEVELS = [
    (-3, 3),   # extreme contrast
    (-1, 1),   # moderate contrast
    (3, 2),    # strong vs moderate positive
    (-3, -2),  # strong vs moderate negative
    (1, 2),    # moderate vs slight positive
    (-1, -2),  # moderate vs slight negative
]


In [7]:
import random

def make_pairs_for_group(group, pair_levels=PAIR_LEVELS, randomize_order=True):
    records = []
    scenario_id = group["original_index"].iloc[0]
    dimension = group["gps_dimension"].iloc[0]
    prompt = group["prompt"].iloc[0]

    by_level = {
        int(row["target_level"]): row
        for _, row in group.iterrows()
        #if row["qc_status"] == "pass"
    }

    for left_level, right_level in pair_levels:
        if left_level not in by_level or right_level not in by_level:
            continue

        left = by_level[left_level]
        right = by_level[right_level]

        option_1 = {
            "level": left_level,
            "response": left["generated_response"]
        }
        option_2 = {
            "level": right_level,
            "response": right["generated_response"]
        }

        if randomize_order and random.random() < 0.5:
            option_1, option_2 = option_2, option_1

        records.append({
            "scenario_id": scenario_id,
            "gps_dimension": dimension,
            "prompt": prompt,
            "response_A": option_1["response"],
            "response_B": option_2["response"],
            "response_A_level": option_1["level"],
            "response_B_level": option_2["level"],
            "pair_type": f"{left_level}_vs_{right_level}"
        })

    return records


In [8]:
pair_records = []

for _, group in df.groupby(["original_index", "gps_dimension"]):
    pair_records.extend(make_pairs_for_group(group))

paired_df = pd.DataFrame(pair_records)

paired_df.to_csv(path / 'paired_comparison_items.csv', index=False)
paired_df.head()


,scenario_id,gps_dimension,prompt,response_A,response_B,response_A_level,response_B_level,pair_type
0,0,risk_taking,Context: As I prepare to launch my own busines...,I choose the proven marketing strategy. It ens...,I go for the innovative campaign. The potentia...,-3,3,-3_vs_3
1,0,risk_taking,Context: As I prepare to launch my own busines...,I opt for the proven marketing strategy. While...,"I lean towards the innovative campaign, but wi...",-1,1,-1_vs_1
2,0,risk_taking,Context: As I prepare to launch my own busines...,"I lean towards the innovative campaign, as the...",I go for the innovative campaign. The potentia...,2,3,3_vs_2
3,0,risk_taking,Context: As I prepare to launch my own busines...,I opt for the proven marketing strategy. While...,I choose the proven marketing strategy. It ens...,-2,-3,-3_vs_-2
4,0,risk_taking,Context: As I prepare to launch my own busines...,"I lean towards the innovative campaign, as the...","I lean towards the innovative campaign, but wi...",2,1,1_vs_2


In [9]:
def score_to_target_level(score):
    """
    Convert a -1 to 1 trait score to the generated response scale [-2, 2].
    """
    return score * 3


def score_to_discrete_level(score):
    """
    Convert a -1 to 1 trait score to one of {-2, -1, 0, 1, 2}.
    """
    return round(score * 3)


def score_to_binned_level(score):
    if score < -0.66:
        return -3

    elif score < -0.33:
        return -2

    elif score < 0.0:
        return -1

    elif score < 0.33:
        return 1

    elif score < 0.66:
        return 2

    else:
        return 3


def choose_closest_option(row):
    target = row["target_level"]
    a_level = row["response_A_level"]
    b_level = row["response_B_level"]

    a_distance = abs(a_level - target)
    b_distance = abs(b_level - target)

    if a_distance < b_distance:
        return "A"
    elif b_distance < a_distance:
        return "B"
    else:
        return "tie"


def choose_closest_option_with_conservative_tie(row):
    target = row["target_level"]
    a_level = row["response_A_level"]
    b_level = row["response_B_level"]

    a_distance = abs(a_level - target)
    b_distance = abs(b_level - target)

    if a_distance < b_distance:
        return "A"
    elif b_distance < a_distance:
        return "B"
    else:
        # tie-breaker: choose the option furthest from neutral
        if abs(a_level) > abs(b_level):
            return "A"
        elif abs(b_level) > abs(a_level):
            return "B"
        else:
            return "A"


In [10]:
def transform_to_scores(scores, paired_df):
    paired_df = paired_df.copy()
    paired_df["trait_score"] = paired_df["gps_dimension"].map(scores)
    paired_df["target_level"] = paired_df["trait_score"].apply(score_to_binned_level)
    paired_df["chosen_option"] = paired_df.apply(choose_closest_option_with_conservative_tie, axis=1)

    paired_df["chosen"] = paired_df.apply(
        lambda row: row["response_A"] if row["chosen_option"] == "A"
        else row["response_B"] if row["chosen_option"] == "B"
        else None,
        axis=1)

    paired_df["rejected"] = paired_df.apply(
        lambda row: row["response_B"] if row["chosen_option"] == "A"
        else row["response_A"] if row["chosen_option"] == "B"
        else None,
        axis=1)

    paired_df["chosen_level"] = paired_df.apply(
        lambda row: row["response_A_level"] if row["chosen_option"] == "A"
        else row["response_B_level"] if row["chosen_option"] == "B"
        else None,
        axis=1)

    paired_df["rejected_level"] = paired_df.apply(
        lambda row: row["response_B_level"] if row["chosen_option"] == "A"
        else row["response_A_level"] if row["chosen_option"] == "B"
        else None,
        axis=1)
    return paired_df


In [13]:
for country_code in COUNTRY_CODES:
    raw_file, triplets_csv, triplets_jsonl, data_country_code, adapter_tag = build_country_paths(country_code)

    scores = country_frames[country_code].groupby('gps_dimension')['z_value'].mean().to_dict()
    country_scores[country_code] = scores

    country_pairs = transform_to_scores(scores, paired_df)

    country_pairs.to_csv(triplets_csv, index=False)
    country_pairs.to_json(triplets_jsonl, orient='records', lines=True)

    print(f'{country_code}: wrote {len(country_pairs)} triplets to {triplets_csv}')
    print(f'{country_code}: wrote {len(country_pairs)} triplets to {triplets_jsonl}')


US: wrote 3594 triplets to /content/drive/MyDrive/DPO/pre-processing/US_triplets_3594_3.csv
US: wrote 3594 triplets to /content/drive/MyDrive/DPO/pre-processing/US_triplets_3594_3.jsonl
MEX: wrote 3594 triplets to /content/drive/MyDrive/DPO/pre-processing/MEX_triplets_3594_3.csv
MEX: wrote 3594 triplets to /content/drive/MyDrive/DPO/pre-processing/MEX_triplets_3594_3.jsonl
